# 1. Importación

Carga de las librerías necesarias y del dataset de features resultante de `02_feature_engineering.ipynb` (`data/processed/valencia/listings_full_features.csv`).

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [2]:
df = pd.read_csv("/Users/yagocoll/Documents/Master/Airbnb/data/processed/valencia/listings_full_features.csv")
df.shape

(7234, 89)

## 2. Preparación de X e y

Separar identificadores, objetivo (`price`) y features. Convertir las columnas booleanas a 0/1 y decidir qué hacer con los nulos que quedan, ya que una regresión lineal no admite `NaN` directamente.

### 2.1 Identificadores, objetivo y features

Además de los identificadores, hay que excluir de `feature_cols` las columnas calculadas directamente a partir de `price` (`price_log`, `price_per_accommodate`, `price_per_min_night`): si se dejan dentro, el modelo no aprende ningún patrón real, deshace la fórmula y "adivina" el precio exacto. Es fuga de información, igual que en Barcelona/Madrid.

**Fuga corregida más abajo**: `neighbourhood_price_encoded` (creada en `02_feature_engineering.ipynb`) se calculó allí usando todo el dataset, no solo lo que aquí es train. Se arregla en la sección 3.1bis, justo después del split. Por eso `neighbourhood_cleansed` (la columna cruda) sigue en `df` en este punto.

In [3]:
id_cols = ["id", "host_id", "host_profile_id"]
target_col = "price"
leakage_cols = ["price_log", "price_per_accommodate", "price_per_min_night"]
# neighbourhood_cleansed todavía no es una feature (pendiente de la corrección de la
# sección 3.1bis): se excluye de X igual que los identificadores, pero se mantiene en
# df para poder usarla justo después del split.
pending_cols = ["neighbourhood_cleansed"]
feature_cols = [c for c in df.columns if c not in id_cols + [target_col] + leakage_cols + pending_cols]

X = df[feature_cols].copy()
y = df[target_col].copy()

X.shape, y.shape

((7234, 81), (7234,))

### 2.2 Booleanas a 0/1

50 de las 81 features son booleanas (los one-hot y los flags) — más que en Barcelona, por los 19 dummies de distrito frente a los 10 de Barcelona. `scikit-learn` las admite tal cual, pero se convierten a `int` de forma explícita.

In [4]:
bool_cols = X.select_dtypes(include="bool").columns
X[bool_cols] = X[bool_cols].astype(int)
len(bool_cols)

50

### 2.3 Nulos restantes

`review_scores_rating`, `listing_age_days` y `days_since_last_review` son `NaN` en las 791 filas sin reviews todavía (`has_reviews == False`). Para este baseline se imputan con la mediana; un modelo de árboles en `04_model_training.ipynb` podrá trabajar con el `NaN` directamente.

In [5]:
null_cols = X.columns[X.isnull().any()].tolist()
print(null_cols)

X[null_cols] = X[null_cols].fillna(X[null_cols].median())
X.isnull().sum().sum()

['review_scores_rating', 'listing_age_days', 'days_since_last_review']


np.int64(0)

`X` queda con 81 columnas numéricas sin nulos, e `y` es `price` sin transformar (el logaritmo se aplica más adelante, solo para el modelo de la sección 6).

## 3. Train/test split

Reservar un conjunto de test antes de tocar nada más, y guardarlo en `data/processed/valencia/` para que `04_model_training.ipynb` y `05_model_evaluation.ipynb` partan del mismo split.

### 3.1 Dividir

80/20, con `random_state` fijo. Se estratifica por `room_type`: en la EDA se vio que `Shared room` (0.25%) y `Hotel room` (0.08%) tienen muestra muy escasa en Valencia, todavía más residual que en Barcelona. Un split aleatorio sin más podría dejar a alguna de las dos con muy pocas filas en test por puro azar.

In [6]:
room_type_cols = [c for c in df.columns if c.startswith("room_type_")]
room_type_for_stratify = df[room_type_cols].idxmax(axis=1)

X_train, X_test, y_train, y_test, df_train, df_test = train_test_split(
    X, y, df, test_size=0.2, random_state=42, stratify=room_type_for_stratify
)

X_train.shape, X_test.shape

((5787, 81), (1447, 81))

### 3.1bis Corregir la fuga de `neighbourhood_price_encoded`

Mismo cálculo que en `02_feature_engineering.ipynb` (media de `price` por `neighbourhood_cleansed`, suavizada con la media global y `smoothing=10`), pero ahora solo con `df_train`. El mapa aprendido en train se aplica tal cual a `df_test` (un barrio de test que no apareciera en train recibiría la media global de train como respaldo; con 84 barrios y ~5800 filas de train, en la práctica no se da el caso, se comprueba abajo).

Con esto corregido, `neighbourhood_cleansed` ya cumplió su función y se descarta de `df_train`/`df_test`.

In [7]:
smoothing = 10
global_mean_price_train = y_train.mean()
neigh_stats_train = df_train.groupby("neighbourhood_cleansed")["price"].agg(["mean", "count"])
smoothed_mean_train = (
    neigh_stats_train["count"] * neigh_stats_train["mean"] + smoothing * global_mean_price_train
) / (neigh_stats_train["count"] + smoothing)

df_train["neighbourhood_price_encoded"] = df_train["neighbourhood_cleansed"].map(smoothed_mean_train)
df_test["neighbourhood_price_encoded"] = (
    df_test["neighbourhood_cleansed"].map(smoothed_mean_train).fillna(global_mean_price_train)
)

print("barrios de test no vistos en train:", df_test["neighbourhood_cleansed"].map(smoothed_mean_train).isnull().sum())

# X_train/X_test ya tenían la versión con fuga (calculada en la sección 2.1 antes del
# split): se sincronizan con el valor corregido de df_train/df_test.
X_train["neighbourhood_price_encoded"] = df_train["neighbourhood_price_encoded"]
X_test["neighbourhood_price_encoded"] = df_test["neighbourhood_price_encoded"]

df_train = df_train.drop(columns=["neighbourhood_cleansed"])
df_test = df_test.drop(columns=["neighbourhood_cleansed"])

df_train[["neighbourhood_price_encoded"]].describe()

barrios de test no vistos en train: 0


,neighbourhood_price_encoded
count,5787.000000
mean,169.889277
std,26.677669
min,106.293321
25%,150.217416
50%,171.098998
75%,189.812047
max,256.798977


Como se esperaba, los 84 barrios de Valencia aparecen todos en train, ningún barrio de test se queda sin mapa.

### 3.2 Guardar el split

Se guarda `df_train`/`df_test` ya con `neighbourhood_price_encoded` corregido, pero antes de la imputación y la conversión de booleanas de la sección 2, para que `04_model_training.ipynb` y `05_model_evaluation.ipynb` puedan decidir su propio tratamiento.

In [8]:
df_train.to_csv("/Users/yagocoll/Documents/Master/Airbnb/data/processed/valencia/listings_train.csv", index=False)
df_test.to_csv("/Users/yagocoll/Documents/Master/Airbnb/data/processed/valencia/listings_test.csv", index=False)

## 4. Baseline ingenuo

Un modelo trivial (predecir siempre la media, la mediana, o la mediana por una variable de tamaño) como suelo mínimo: cualquier modelo real tiene que superar esto para que merezca la pena.

### 4.1 Predecir siempre la media

Por definición, un modelo que siempre predice la media del train tiene R² ≈ 0 sobre el test. Sirve como punto cero.

In [9]:
global_mean = y_train.mean()
pred_mean = np.full(len(y_test), global_mean)

print("RMSE:", np.sqrt(mean_squared_error(y_test, pred_mean)))
print("MAE:", mean_absolute_error(y_test, pred_mean))
print("R2:", r2_score(y_test, pred_mean))

RMSE: 139.4547518738749
MAE: 73.99871889535311
R2: -1.6207135682844154e-05


### 4.2 Predecir siempre la mediana

Dado el sesgo de `price` (skew 27 en la EDA, todavía más marcado que en Barcelona), la mediana debería ser un mejor "valor típico" que la media.

In [10]:
global_median = y_train.median()
pred_median = np.full(len(y_test), global_median)

print("RMSE:", np.sqrt(mean_squared_error(y_test, pred_median)))
print("MAE:", mean_absolute_error(y_test, pred_median))
print("R2:", r2_score(y_test, pred_median))

RMSE: 140.54618852036484
MAE: 72.01543192812716
R2: -0.01573062964403471


El MAE mejora (72.0€ frente a 74.0€ con la media), pero el R² empeora (-0.02 frente a ~0.00): el R² compara contra la media por definición, así que cualquier predicción distinta puede bajarlo aunque sea mejor en otros términos. Igual que en Barcelona, un recordatorio de que la métrica de referencia importa.

### 4.3 Predecir la mediana según `accommodates`

A diferencia de Barcelona (donde `bedrooms` fue la variable ganadora, corr. 0.66), en la EDA/feature engineering de Valencia la variable más relacionada con `price` es **`accommodates`** (Spearman 0.63, Pearson 0.35). Un baseline algo menos ingenuo: mediana por cada valor de `accommodates`, calculada solo con train.

In [11]:
train_medians_by_accommodates = X_train.assign(price=y_train).groupby("accommodates")["price"].median()

pred_accommodates = X_test["accommodates"].map(train_medians_by_accommodates).fillna(global_median)

print("RMSE:", np.sqrt(mean_squared_error(y_test, pred_accommodates)))
print("MAE:", mean_absolute_error(y_test, pred_accommodates))
print("R2:", r2_score(y_test, pred_accommodates))

RMSE: 123.70790419644682
MAE: 56.04860400829302
R2: 0.21307144858879945


Mejora clara: RMSE de 140.5 a 123.7, MAE de 72.0 a 56.0, R² de -0.02 a **0.21**. Es una mejora mucho más modesta que la de Barcelona (que llegaba a 0.46 agrupando por `bedrooms`): en Valencia, ni siquiera la mejor variable de tamaño por sí sola explica tanto de la varianza de `price`, coherente con las correlaciones más débiles ya vistas en la EDA. Aun así, pone un listón real: cualquier modelo tiene que superar claramente este 0.21.

## 5. Métricas de evaluación

Definir aquí las métricas que se van a usar de forma consistente en todo el modelado (RMSE, MAE, R², MAPE).

### 5.1 Función `evaluate`

Se añade una cuarta métrica, MAPE, el error medio en porcentaje sobre el precio real. Todo se empaqueta en una función para no repetir el código en cada modelo.

In [12]:
def evaluate(y_true, y_pred, name):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    return {"modelo": name, "RMSE": rmse, "MAE": mae, "R2": r2, "MAPE": mape}

### 5.2 Tabla comparativa de los baselines

In [13]:
rows = [
    evaluate(y_test, pred_mean, "Media"),
    evaluate(y_test, pred_median, "Mediana"),
    evaluate(y_test, pred_accommodates, "Mediana por accommodates"),
]

results = pd.DataFrame(rows).set_index("modelo")
results.round(2)

,RMSE,MAE,R2,MAPE
modelo,,,,
Media,139.45,74.00,-0.00,88.13
Mediana,140.55,72.02,-0.02,77.56
Mediana por accommodates,123.71,56.05,0.21,45.30


El MAPE sale muy alto (77-88%), igual que en Barcelona y por el mismo motivo: hay anuncios muy baratos (`price` mínimo 1.87€) donde un error de pocos euros ya es un porcentaje enorme. Conviene fiarse más de RMSE/MAE/R² que del MAPE por sí solo.

## 6. Baseline real: regresión lineal

Un primer modelo simple e interpretable, entrenado sobre `price_log` por el sesgo ya visto en la EDA.

### 6.1 Entrenar

Se entrena sobre `log1p(price)`. Las predicciones se deshacen con `expm1` antes de evaluar.

In [14]:
y_train_log = np.log1p(y_train)

model = LinearRegression()
model.fit(X_train, y_train_log)

pred_log = model.predict(X_test)
pred_lr = np.expm1(pred_log)

/Users/yagocoll/Documents/Master/Airbnb/.venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/yagocoll/Documents/Master/Airbnb/.venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/Users/yagocoll/Documents/Master/Airbnb/.venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: invalid value encountered in matmul
  return X @ coef_ + self.intercept_


### 6.2 Evaluar

In [15]:
results.loc["Regresión lineal (log)"] = evaluate(y_test, pred_lr, "Regresión lineal (log)")
results.round(2)

,RMSE,MAE,R2,MAPE
modelo,,,,
Media,139.45,74.00,-0.00,88.13
Mediana,140.55,72.02,-0.02,77.56
Mediana por accommodates,123.71,56.05,0.21,45.30
Regresión lineal (log),114.70,45.55,0.32,29.10


**R²=0.32, MAE≈45.5€**: mejora sobre el mejor baseline ingenuo (R²=0.21), pero mucho más modesta que el R²=0.69-0.77 que alcanzaba la regresión lineal en Barcelona. Con 81 features y un modelo real todavía se explica bastante menos de un tercio de la varianza. Coherente con toda la EDA: `price` en Valencia tiene una cola mucho más larga (skew 27 vs. 9 en Barcelona) y las correlaciones lineales de las variables de tamaño son más débiles, así que un modelo lineal simple tiene de partida menos margen aquí. Esto no es un fallo del pipeline: es la misma conclusión de la sección 10 de `02_feature_engineering.ipynb` (Spearman >> Pearson en Valencia) reapareciendo ahora en el propio rendimiento del modelo. Un modelo no lineal (árboles, en `04_model_training.ipynb`) es más prometedor aquí que en Barcelona, precisamente porque no depende de relaciones lineales.

### 6.3 Un aviso a tener en cuenta

Al entrenar aparecen avisos de `numpy` (`divide by zero`, `overflow`... `encountered in matmul`), igual que en Barcelona.

In [16]:
import numpy.linalg as la

la.cond(X_train.values)

np.float64(1.5099831723722785e+19)

Un número de condición altísimo (aún mayor que en Barcelona) indica una matriz muy mal condicionada: `X` incluye a propósito tanto la versión bruta como la versión `_log` de varias variables (`bedrooms`/`bedrooms_log`...) y varias codificaciones categóricas que se solapan. Esto no invalida las métricas (`scikit-learn` resuelve con SVD, sin `NaN`/`inf` en las predicciones), pero sí impide interpretar los coeficientes uno a uno. Se deja igual que en Barcelona para `04_model_training.ipynb` (modelo regularizado o selección de variables, si hiciera falta interpretar coeficientes).

## 7. Conclusiones

Resumen de los resultados del baseline.

### Resultados

| Modelo | RMSE | MAE | R² | MAPE |
|---|---|---|---|---|
| Media | 139.45 | 74.00 | -0.00 | 88.13 |
| Mediana | 140.55 | 72.02 | -0.02 | 77.56 |
| Mediana por `accommodates` | 123.71 | 56.05 | 0.21 | 45.30 |
| Regresión lineal (log) | 114.70 | 45.55 | 0.32 | 29.10 |

Cada paso mejora sobre el anterior, pero con márgenes más ajustados que en Barcelona: agrupar por `accommodates` ya recorta el MAE de 74€ a 56€, y el modelo real con las 81 variables lo baja a 45.5€, aunque el R² se queda en 0.32 frente al ~0.7 de Barcelona/Madrid.

### Tres problemas encontrados y cómo se trataron

- **Fuga de información directa**: `price_log`, `price_per_accommodate` y `price_per_min_night` estaban calculadas a partir de `price` y se habían colado como features. Se excluyeron en la sección 2, igual que en Barcelona/Madrid.
- **Fuga más leve, ahora corregida**: `neighbourhood_price_encoded` se calculaba con todo el dataset. Se corrige en la sección 3.1bis, recalculándola solo con `df_train`.
- **Muestra escasa en `Shared room`/`Hotel room`** (18 y 6 anuncios sobre 7234, todavía más residual que en Barcelona): el split estratifica por `room_type`.
- **Multicolinealidad severa** en la regresión lineal, por la versión bruta y `_log` de varias variables a la vez. No afecta a las métricas de predicción, pero impide interpretar los coeficientes uno a uno.

### Una diferencia real con Barcelona/Madrid, no un error de pipeline

El rendimiento del baseline lineal en Valencia (R²=0.32) es notablemente más bajo que en Barcelona. La causa se puede rastrear hasta la EDA: `price` tiene una cola mucho más larga en Valencia (skew 27 vs. 9), las correlaciones de Pearson de las variables de tamaño son bastante más débiles que las de Spearman (p. ej. `accommodates` 0.35 vs. 0.63), y `distance_to_center_km` no aporta nada por el patrón de doble polo (centro + playa) de la ciudad. Todo apunta en la misma dirección: la relación entre las variables y `price` en Valencia es más real (existe, y con fuerza según Spearman) pero menos **lineal** que en Barcelona. Esto hace de Valencia un caso especialmente interesante para comparar modelos lineales contra modelos de árboles en `04_model_training.ipynb`.

### El listón para `04_model_training.ipynb`

Cualquier modelo más complejo (Random Forest, XGBoost, LightGBM...) tiene que superar claramente **R²=0.32 / MAE≈45.5€**. Un modelo de árboles no necesita la imputación por mediana de la sección 2.3, no le afecta la multicolinealidad de la sección 6.3, y al no asumir relaciones lineales debería beneficiarse más aquí que en Barcelona de las variables cuya relación con `price` es fuerte mas no lineal (`accommodates`, `has_license`, `neighbourhood_price_encoded`).

### Lo que queda guardado

`listings_train.csv` y `listings_test.csv` en `data/processed/valencia/`, con el mismo split (80/20, `random_state=42`, estratificado por `room_type`) y `neighbourhood_price_encoded` ya corregido, para que `04_model_training.ipynb` y `05_model_evaluation.ipynb` trabajen sobre las mismas filas y los resultados sean comparables entre notebooks.